# 영화 데이터로 Python 복습

KOBIS(영화관입장권통합전산망) 박스오피스 API로 영화 데이터를 가져오고, 보고 싶은 정보를 코드로 만들어 봅니다.

| 단계 | 내용 |
| --- | --- |
| 0. 같이 보기 | API 구조 · 하루치 데이터 살펴보기 |
| 1. 하루 데이터로 분석 | 10월 1일 하루 |
| 2. 여러 날 데이터로 분석 | 올해 데이터를 쌓아서 |
| 3. API 활용하여 분석 | 상세정보를 더 받아 붙이기 |
| 4. 내가 보고 싶은 정보 | 질문을 직접 정하고 코드로 |
| ★ 심화 | 배우 검색 · 요청 줄이기 · 모듈로 옮기기 |

문제마다 빈 칸을 채우고 아래 칸을 실행해 **출력 모양**과 비교합니다. `(먼저)`가 붙은 문제는 뒤 문제가 그 결과를 쓰니 먼저 풉니다. ★은 앞을 끝낸 뒤에 풉니다.

> 받은 데이터는 이 노트북과 `data/` 폴더 안에서만 씁니다. 다른 사람에게 보내거나 올리지 않고, 프로젝트가 끝나면 지웁니다(KOBIS 이용약관).

## 0. 같이 보기

`.env`에 키를 넣었는지 확인합니다(README "키 넣기").

요청 주소는 `kobis.py`의 `URLMaker`가 만들어 줍니다. 사용법은 `docs/kobis.md`에 있습니다.

In [ ]:
import requests

from kobis import URLMaker, load_key

url_maker = URLMaker(load_key())

2026년 10월 1일 일별 박스오피스를 요청합니다.

주소에는 내 키가 들어 있습니다. 주소와 요청 에러 메시지는 화면에 띄우거나 Claude에 붙여 넣지 않습니다.

In [ ]:
url = url_maker.daily_boxoffice_url("20261001")
response = requests.get(url, timeout=10)
response.status_code

In [ ]:
data = response.json()
data.keys()

In [ ]:
data["boxOfficeResult"].keys()

In [ ]:
data["boxOfficeResult"]["showRange"]

In [ ]:
movies = data["boxOfficeResult"]["dailyBoxOfficeList"]
len(movies)

In [ ]:
movies[0]

숫자도 모두 문자열(`"55008"`)로 옵니다. 필드마다 뜻은 `docs/kobis.md`의 `daily_boxoffice_url` 표에 있습니다.

In [ ]:
movies[0]["movieNm"], movies[0]["audiCnt"], movies[0]["rankInten"]

## 1. 하루 데이터로 분석

여기부터 혼자 풉니다. 0에서 받은 `data` · `movies`를 씁니다.

### 1-1. 영화 목록 꺼내기

응답 전체를 받아 영화 목록(리스트)을 돌려줍니다. 2단계에서 날짜마다 다시 씁니다.

**출력 모양**
```
10
```

In [ ]:
def movie_list(data: dict) -> list:
    pass

In [ ]:
len(movie_list(data))

### 1-2. 그날 관객 합계

영화 목록을 받아 그날 관객 수(`audiCnt`) 합계를 숫자로 돌려줍니다.

- `audiCnt`는 문자열입니다
- `f"{숫자:,}"`는 천 단위에 쉼표를 붙입니다. `f"{1234567:,}"` → `1,234,567`

**출력 모양**
```
총 123,456명
```

In [ ]:
def total_audience(movies: list) -> int:
    pass

In [ ]:
print(f"총 {total_audience(movies):,}명")

### 1-3. 순위 변동으로 나누기

영화 목록을 받아 전날과 비교한 순위 변동으로 나누고, 제목을 딕셔너리에 모아 돌려줍니다.

| 키 | 조건 |
| --- | --- |
| `"새로"` | `rankOldAndNew`가 `"NEW"` |
| `"오름"` | `rankInten`이 양수 |
| `"내림"` | `rankInten`이 음수 |
| `"그대로"` | `rankInten`이 0 |

- 새로 들어온 영화도 `rankInten`이 `"0"`입니다. `"NEW"`인지 먼저 봅니다
- 해당하는 영화가 없어도 네 키는 모두 있게 합니다

**출력 모양**
```
{'새로': [], '오름': ['영화제목', ...], '내림': ['영화제목'], '그대로': ['영화제목', ...]}
```

In [ ]:
def group_by_change(movies: list) -> dict:
    pass

In [ ]:
group_by_change(movies)

확인: 세 줄 모두 `True`면 맞습니다. 세 번째 줄은 새로 들어온 영화 하나를 직접 만들어 넣어 봅니다.

In [ ]:
groups = group_by_change(movies)
count = 0
for titles in groups.values():
    count += len(titles)

print(set(groups.keys()) == {"새로", "오름", "내림", "그대로"})
print(count == len(movies))
print(group_by_change([{"movieNm": "시험", "rankOldAndNew": "NEW", "rankInten": "0"}])["새로"] == ["시험"])

### 1-4. 필요한 칸만 새로 (먼저)

영화 목록을 받아, 영화마다 아래 칸만 가진 새 딕셔너리를 만들어 리스트로 돌려줍니다. 숫자는 `int`로 바꿉니다.

| 새 키 | KOBIS 필드 |
| --- | --- |
| `"movie_cd"` | `movieCd` |
| `"title"` | `movieNm` |
| `"open_date"` | `openDt` |
| `"audience"` | `audiCnt` |
| `"audience_acc"` | `audiAcc` |
| `"sales_acc"` | `salesAcc` |

뒤의 문제는 이 함수가 만든 모양으로 풉니다.

**출력 모양**
```
{'movie_cd': '20255033', 'title': '영화제목', 'open_date': '2026-09-23', 'audience': 55008, 'audience_acc': 1821847, 'sales_acc': 17848865130}
```

In [ ]:
def to_rows(movies: list) -> list:
    pass

In [ ]:
rows = to_rows(movies)
rows[0]

확인: 세 줄 모두 `True`면 맞습니다.

In [ ]:
print(len(rows) == len(movies))
print(set(rows[0].keys()) == {"movie_cd", "title", "open_date", "audience", "audience_acc", "sales_acc"})
print(type(rows[0]["audience"]) == int and type(rows[0]["audience_acc"]) == int and type(rows[0]["sales_acc"]) == int)

### 1-5. ★ 상영 1회당 관객

관객 수 1위 영화가 상영 한 번에도 관객이 가장 많을까요? 영화마다 상영 1회당 관객(`audiCnt` ÷ `showCnt`)을 구해, 많은 순서로 출력합니다.

- `showCnt`는 그날 상영 횟수입니다
- 몫만 구하면 됩니다(`//`)
- 1-4처럼 영화마다 딕셔너리를 만들어 리스트에 모으고, `sorted(key=lambda …)`로 정렬합니다

**출력 모양**
```
1. 영화제목 상영 1,234회 · 1회당 25명
2. 영화제목 상영 3,095회 · 1회당 17명
...
```

## 2. 여러 날 데이터로 분석

"올해 가장 흥행한 영화"는 하루치로 답할 수 없습니다. 날마다 받아야 합니다.

2단계에서 쓰는 것을 가져옵니다. 커널을 다시 시작했으면 0의 첫 칸, 1단계 함수 칸, 이 칸을 다시 실행합니다.

In [ ]:
import json
from datetime import date, timedelta
from pathlib import Path

folder = Path("data/daily")
folder.mkdir(parents=True, exist_ok=True)

`mkdir(parents=True, exist_ok=True)`는 `data/daily` 폴더를 만들고, 이미 있으면 그냥 넘어갑니다.

### 요청으로 바로 일주일

1월 1일부터 7일까지 날마다 요청해서 1위를 출력합니다. 날짜는 05장 `datetime`으로 만듭니다. 1-1의 `movie_list`를 그대로 씁니다.

In [ ]:
start = date(2026, 1, 1)
for i in range(7):
    target_dt = (start + timedelta(days=i)).strftime("%Y%m%d")
    day_data = requests.get(url_maker.daily_boxoffice_url(target_dt), timeout=10).json()
    print(target_dt, movie_list(day_data)[0]["movieNm"])

`timedelta`로 하루씩 더하면 1월 31일 다음이 2월 1일이 됩니다.

요청만으로도 여러 날을 볼 수 있습니다. 올해 전체는 요청이 304번이라 질문을 바꿀 때마다 기다릴 수 없습니다. 한 번 받아 파일로 저장하고, 그다음부터는 파일을 읽습니다.

### 하루를 파일로

1월 1일 하루를 받아 `data/daily/20260101.json`에 저장합니다.

In [ ]:
target_dt = "20260101"
day_data = requests.get(url_maker.daily_boxoffice_url(target_dt), timeout=10).json()

path = folder / f"{target_dt}.json"
with open(path, "w", encoding="utf-8") as f:
    json.dump(day_data, f, ensure_ascii=False)

print("저장:", path)

왼쪽 파일 목록에서 `data/daily/20260101.json`을 열어 봅니다. 0에서 본 `data`와 같은 모양입니다.

### 2-0. 파일 하나 읽기

날짜를 받아 `data/daily/날짜.json`을 읽고, 응답 전체(딕셔너리)를 돌려줍니다(08장).

저장한 것과 읽은 것이 같으면 아래 칸이 `True`입니다.

**출력 모양**
```
True
```

In [ ]:
def load_day(target_dt: str) -> dict:
    pass

In [ ]:
load_day("20260101") == day_data

요청한 결과와 파일에서 읽은 결과가 같은 모양이라 `movie_list` · `to_rows`를 그대로 씁니다.

### 올해 전체 쌓기

일주일 칸처럼 날짜를 하루씩 늘리면서, 하루를 파일로 저장한 코드를 반복합니다. 1월 1일부터 10월 31일까지입니다. 아래 칸을 그대로 실행합니다.

- 요청이 304번 갑니다. 5분쯤 걸립니다. 기다리는 동안 다음 문제를 읽습니다
- 파일이 이미 있는 날짜는 건너뜁니다
- 요청이 실패하면 멈추고 날짜를 출력합니다. 다시 실행하면 이어서 받습니다

In [ ]:
start = date(2026, 1, 1)
end = date(2026, 10, 31)

day = start
while day <= end:
    target_dt = day.strftime("%Y%m%d")
    path = folder / f"{target_dt}.json"

    if not path.exists():
        try:
            day_data = requests.get(url_maker.daily_boxoffice_url(target_dt), timeout=10).json()
        except requests.RequestException:
            day_data = {}
        if "boxOfficeResult" not in day_data:
            print("요청 실패:", target_dt, "다시 실행하면 여기서부터 이어서 받습니다")
            break
        with open(path, "w", encoding="utf-8") as f:
            json.dump(day_data, f, ensure_ascii=False)

    if target_dt.endswith("01"):
        print(target_dt[:6], "받는 중")
    day += timedelta(days=1)

print("저장한 날:", len(list(folder.glob("*.json"))))

날짜 목록을 만듭니다. 출력이 `304`가 아니면 위 칸을 다시 실행합니다. `folder.glob("*.json")`은 폴더 안의 `.json` 파일 경로를 모두 돌려주고, `path.stem`은 확장자를 뺀 이름(`"20260101"`)입니다.

In [ ]:
dates = []
for path in sorted(folder.glob("*.json")):
    dates.append(path.stem)

print(len(dates), dates[0], dates[-1])

### 2-1. 날짜별 1위

날짜 목록을 받아 날마다 1위 영화 제목을 출력합니다. `load_day` · `movie_list`를 씁니다.

**출력 모양** (앞 다섯 날)
```
20260101 영화제목
20260102 영화제목
...
```

In [ ]:
def print_first(dates: list):
    pass

In [ ]:
print_first(dates[:5])

### 2-2. 올해 가장 흥행한 영화 (먼저)

날짜 목록을 받아, 올해 TOP 10에 한 번이라도 오른 영화를 모으고 **누적 관객이 많은 순서**로 돌려줍니다.

- 날마다 `to_rows`로 바꿉니다
- 영화 코드(`movie_cd`)를 키로 하는 딕셔너리에 넣습니다. 같은 영화가 여러 날 나오면 뒤의 날짜 값으로 덮어써서, 마지막으로 TOP 10에 있던 날의 누적 관객이 남습니다
- 딕셔너리의 값만 꺼내 `audience_acc`로 정렬합니다. `sorted(리스트, key=lambda row: row["audience_acc"], reverse=True)`(04장)

돌려주는 값은 1-4 `to_rows`와 같은 모양의 딕셔너리 리스트입니다.

**출력 모양**
```
1. 영화제목 (2026-02-04) 12,345,678명
2. 영화제목 (2025-12-17) 6,543,210명
...
```

In [ ]:
def yearly_ranking(dates: list) -> list:
    pass

In [ ]:
ranking = yearly_ranking(dates)

for i, row in enumerate(ranking[:10]):
    print(f"{i + 1}. {row['title']} ({row['open_date']}) {row['audience_acc']:,}명")

확인: 두 줄 모두 `True`면 맞습니다. 첫 줄은 같은 영화가 두 번 들어가지 않았는지, 둘째 줄은 정렬이 맞는지 봅니다.

In [ ]:
codes = []
for row in ranking:
    codes.append(row["movie_cd"])

print(len(codes) == len(set(codes)))
print(ranking[0]["audience_acc"] >= ranking[-1]["audience_acc"])

결과에 2025년에 개봉한 영화가 있나요? 누적 관객에는 작년 관객도 들어 있습니다.

`movie_cd` 대신 제목을 키로 묶으면 결과가 달라질까요? 직접 바꿔 보고, `len(ranking)`을 비교해 봅니다.

### 2-3. 올해 개봉한 영화만

2-2의 `ranking`에서 **2026년에 개봉한** 영화만 골라 `released`에 담고, TOP 10을 출력합니다. `released`는 2-4와 3단계에서 다시 씁니다.

**출력 모양**
```
1. 영화제목 (2026-02-04) 12,345,678명
...
```

In [ ]:
#> ranking에서 2026년에 개봉한 영화만 골라 released에 담고, TOP 10을 출력합니다

확인: 두 줄 모두 `True`면 맞습니다.

In [ ]:
all_2026 = True
for row in released:
    if not row["open_date"].startswith("2026"):
        all_2026 = False

print(len(released) > 0)
print(all_2026)

### 2-4. 평균 관람료

2-3의 `released`에서 영화마다 평균 관람료(누적 매출 ÷ 누적 관객)를 구해, 가장 비싼 세 편과 가장 싼 세 편을 출력합니다.

- 관객이 너무 적은 영화는 빼고, 누적 관객 1만 명 이상만 봅니다
- 몫만 구합니다(`//`)

**출력 모양**
```
비싼 영화
영화제목 31,891원
...
싼 영화
영화제목 5,000원
...
```

가장 비싼 영화들은 어떤 영화인가요? 일반 영화와 무엇이 다를까요?

### 2-5. 1위를 가장 많이 한 영화

올해 1위를 한 날이 가장 많은 영화 다섯 편을 날 수와 함께 출력합니다.

- 날마다 1위 영화 제목을 키로, 1위를 한 날 수를 값으로 세는 딕셔너리를 만듭니다
- `딕셔너리.items()`를 날 수로 정렬합니다. 날 수가 같으면 순서는 상관없습니다

**출력 모양**
```
영화제목 31일
영화제목 18일
...
```

### 2-6. ★ 달마다 가장 많이 본 영화

영화마다 그달 `audiCnt`를 모두 더해, 달마다 가장 많은 영화를 출력합니다. TOP 10에 든 날의 관객만 들어갑니다.

**출력 모양**
```
01월 영화제목 (1,234,567명)
02월 영화제목 (2,345,678명)
...
```

### 2-7. ★ 개봉하고 며칠 만에 1위

올해 1위를 한 영화마다, 개봉하고 며칠째에 처음 1위를 했는지 출력합니다. 처음 1위를 한 날짜 순서로 출력합니다.

- 날짜끼리 빼면 `timedelta`가 나오고, `.days`로 며칠인지 꺼냅니다(05장)
- `"2026-02-04"` · `"20260204"` 같은 문자열은 슬라이싱과 `int()`로 `date(2026, 2, 4)`를 만들 수 있습니다
- 개봉일을 1일째로 셉니다. 개봉한 날 1위면 1일째
- 개봉일이 공백(`" "`)인 영화가 있으면 건너뜁니다

**출력 모양**
```
영화제목 개봉 2025-12-17 → 1위 20260101 (16일째)
영화제목 개봉 2026-02-04 → 1위 20260204 (1일째)
...
```

## 3. API 활용하여 분석

박스오피스 목록에는 감독 · 나라 · 장르가 없습니다. 영화 코드로 상세정보를 더 받아 붙입니다.

### 3-1. 상세정보를 원하는 모양으로 (먼저)

영화 코드를 받아 상세정보를 요청하고, 아래 칸만 가진 딕셔너리로 돌려줍니다. 응답 필드는 `docs/kobis.md`의 `movie_info_url`에 있습니다.

| 새 키 | 응답 필드 | 모양 |
| --- | --- | --- |
| `"title"` | `movieNm` | 문자열 |
| `"year"` | `prdtYear` (제작연도) | 문자열 |
| `"show_time"` | `showTm` | `int`. 비어 있으면 `0` |
| `"directors"` | `directors`의 `peopleNm` | 이름 리스트 |
| `"actors"` | `actors`의 `peopleNm` | 앞의 세 명까지 이름 리스트 |
| `"genres"` | `genres`의 `genreNm` | 이름 리스트 |
| `"nations"` | `nations`의 `nationNm` | 이름 리스트 |

비어 있는 항목은 빈 리스트로 둡니다.

**출력 모양**
```
{'title': '영화제목',
 'year': '2026',
 'show_time': 98,
 'directors': ['감독이름'],
 'actors': ['배우1', '배우2', '배우3'],
 'genres': ['드라마'],
 'nations': ['한국']}
```

In [ ]:
def movie_info(movie_cd: str) -> dict:
    pass

In [ ]:
movie_info(movies[0]["movieCd"])

### 3-2. 올해 흥행 TOP 10의 나라와 감독

2-3의 `released` TOP 10마다 `movie_info`로 상세정보를 받아 `top10` 리스트에 담고, 나라 · 감독을 붙여 출력합니다. 요청이 10번 갑니다.

- 나라 · 감독이 여럿이면 쉼표로 잇습니다. `", ".join(리스트)`는 리스트를 쉼표로 이은 문자열을 만듭니다
- 감독이 비어 있으면 `미정`으로 씁니다
- 2-3을 못 풀었으면 1-4의 `rows`로 풉니다

**출력 모양**
```
1. 영화제목 | 한국 | 감독이름
2. 영화제목 | 미국 | 감독이름
...
```

### 3-3. TOP 10 가운데 한국 영화

3-2의 `top10`으로 답합니다. 요청을 다시 보내지 않습니다.

**출력 모양**
```
한국 영화 N편: 영화제목, 영화제목, ...
```

## 4. 내가 보고 싶은 정보

이 데이터로 보고 싶은 정보를 직접 정하고 코드로 만듭니다. 질문은 마크다운 칸에 한 줄로 적고, 아래 코드 칸에서 답합니다. 마지막 시간에 질문 하나를 골라 보여 줍니다.

질문 예:
- TOP 10에 가장 오래 머문 영화는?
- 개봉 첫날 가장 높은 순위로 들어온 영화는?
- 요일마다 관객 합계는? (`date(2026, 10, 1).weekday()`는 요일 번호. 월요일이 0)
- 올해 흥행 TOP 10의 장르는 무엇이 많은가?

아직 쓰지 않은 필드도 있습니다: `salesShare`(매출 점유율) · `scrnCnt`(스크린 수) · `audiChange`(전날 대비 관객 증감률). `docs/kobis.md` 표를 봅니다.

**질문 1.** 

**질문 2.** 

**질문 3.** 

## ★ 심화

### ★1. 배우 검색

배우 이름을 받아 출연작 다섯 편의 제목 · 개봉일 · 상영시간 · 감독을 출력합니다. `docs/kobis.md`의 `people_list_url` · `people_info_url`을 봅니다.

1. 영화인 목록에서 `filmoNames`의 작품 수가 가장 많은 사람을 고릅니다. 같은 이름이 여럿일 수 있습니다
2. 그 사람의 `filmos`에서 `moviePartNm`이 `"배우"`인 작품을 앞에서 다섯 편 고릅니다
3. 작품마다 `movie_info_url`로 상세정보를 받아 출력합니다. 값이 비어 있으면 `미정`으로 씁니다

**출력 모양**
```
'배우이름'의 출연작 5편

제목: 영화제목
개봉일: 20260930
상영시간: 98분
감독: 감독이름
```

In [ ]:
def search_actor(actor_name: str):
    pass

In [ ]:
search_actor("박정민")

### ★2. 요청 줄이기

3-1의 `movie_info`는 부를 때마다 요청을 보냅니다. 받은 상세정보를 `data/movies/영화코드.json`으로 저장해 두고, 파일이 있으면 요청 없이 파일을 읽는 `movie_info_saved`를 만듭니다. 2단계의 올해 전체 쌓기 칸이 같은 방식입니다.

요청을 보내는 줄 바로 앞에 `print("요청:", movie_cd)`를 넣고 아래 칸을 처음 실행합니다. 같은 영화를 두 번 부르는데 `요청:`이 한 번만 나오면 맞습니다. 다시 실행하면 파일이 이미 있어서 `요청:`이 나오지 않습니다.

In [ ]:
def movie_info_saved(movie_cd: str) -> dict:
    pass

In [ ]:
print(movie_info_saved(movies[1]["movieCd"])["title"])
print(movie_info_saved(movies[1]["movieCd"])["title"])

### ★3. 모듈로 옮기기

`load_day` · `movie_list` · `to_rows` · `yearly_ranking`을 `analysis.py` 파일로 옮깁니다(05장). 함수 안에서 쓰는 것(`import json`, 만들어 둔 변수 등)도 같이 옮깁니다.

다 옮겼으면 커널을 다시 시작하고 아래 칸만 실행합니다. 영화 제목이 나오면 맞습니다. 에러가 나면 메시지를 보고 무엇이 빠졌는지 찾습니다. `analysis.py`를 고친 뒤에도 커널을 다시 시작해야 고친 내용이 반영됩니다.

In [ ]:
from analysis import yearly_ranking

print(yearly_ranking(["20261001", "20261002"])[0]["title"])